# Games for today and tomorrow

Loads your pipeline **games** Parquet and filters rows where `game_date` is today or tomorrow (local timezone).

**Default file:** `data/games_{season}.parquet` where `season` is **`PIPELINE_SEASON`** (if set) or the **calendar year**. Run **`python -m app.main live`** (or Docker **`compose --profile live up live`**) to refresh the current season; use **`backfill --years ...`** for one-off seasons.

Legacy fallbacks: `games.parquet`, `games_live.parquet` (only if present).

**Requires** a schedule window that actually includes *today's* calendar date. If the notebook prints a `game_date` range ending in 2024 but your clock is 2026, the "today / tomorrow" filter will correctly show **0 rows** until you rebuild for the current season.

Run Jupyter with working directory `mlb-model` **or** `mlb-model/notebooks` (the next cell resolves `data/` automatically).

The load cell sets **`display.max_columns` to 50** (raise to **`None`** in that cell to show every column with no cap). The main table shows **all** columns in `sub` for today/tomorrow. A later section flags **mismatch / favorite** games using `MismatchBreakdown`-style thresholds plus optional platoon and bullpen filters.


In [10]:
import os
from datetime import datetime
from pathlib import Path

import pandas as pd

# Show many columns in DataFrame HTML / repr (raise to None for no limit).
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 50)

_cwd = Path.cwd()
DATA = _cwd / "data" if (_cwd / "data").is_dir() else _cwd.parent / "data"

_ps = os.environ.get("PIPELINE_SEASON", "").strip()
_season = int(_ps) if _ps.isdigit() else datetime.now().year

# Prefer games_{season}.parquet; then generic / legacy names. If several exist, use newest mtime.
_candidates = [
    DATA / f"games_{_season}.parquet",
    DATA / "games.parquet",
    DATA / "games_live.parquet",
]
_existing = [p for p in _candidates if p.is_file()]
if not _existing:
    raise FileNotFoundError(
        f"No games parquet in {DATA} (tried games_{_season}.parquet, games.parquet, games_live.parquet)"
    )
PARQUET = max(_existing, key=lambda p: p.stat().st_mtime)
if len(_existing) > 1:
    print("Multiple game parquets found; using newest by file time:", PARQUET.name)

print("Using:", PARQUET.resolve())
df = pd.read_parquet(PARQUET)
df["game_date"] = pd.to_datetime(df["game_date"]).dt.normalize()
gd_min, gd_max = df["game_date"].min(), df["game_date"].max()
print(f"game_date range: {gd_min.date()} .. {gd_max.date()}  ({len(df)} rows)")
if "stats_season" in df.columns:
    ss = df["stats_season"].dropna().unique()
    print("stats_season values:", sorted(ss.tolist()))
clock = pd.Timestamp.today().normalize()
if clock < gd_min or clock > gd_max:
    print(
        f"\n>>> Today's date ({clock.date()}) is outside this file's game_date range.\n"
        "    Re-run the pipeline for the season you care about, e.g. (host):\n"
        "    python -m app.main live --season 2026\n"
        "    Or: docker compose --profile live up live  (set PIPELINE_SEASON if needed)"
    )

Multiple game parquets found; using newest by file time: games_2026.parquet
Using: C:\Users\Austin\baseball\mlb-model\data\games_2026.parquet
game_date range: 2026-03-01 .. 2026-09-27  (2796 rows)
stats_season values: [2026]


In [11]:
# df already loaded and game_date normalized in previous cell
today = pd.Timestamp.today().normalize()
tomorrow = today + pd.Timedelta(days=1)
# sub = df[df["game_date"].isin([today, tomorrow])].copy()
sub = df[df["game_date"].isin([today])].copy()

print(f"Today={today.date()}, tomorrow={tomorrow.date()}, rows={len(sub)}")

Today=2026-04-15, tomorrow=2026-04-16, rows=15


In [12]:
# All columns for today/tomorrow (respects display.max_columns above).
view = sub.sort_values(["game_date", "home_team_name"])

_datapoints = [
    "game_date",
    "home_team_name",
    "away_team_name",
    # batting stats
    "home_wrc_plus",
    "away_wrc_plus",
    "offense_diff",
    "home_offense_platoon",
    "away_offense_platoon",
    "offense_platoon_diff",
    # starting pitcher stats
    "home_probable_pitcher",
    "home_sp_throws",
    "home_sp_kbb",
    "home_sp_kbb_roll14",
    "home_sp_xfip",
    "home_sp_xfip_roll14",
    "away_probable_pitcher",
    "away_sp_throws",
    "away_sp_kbb",
    "away_sp_kbb_roll14",
    "away_sp_xfip",
    "away_sp_xfip_roll14"
    "sp_kbb_diff",
    "sp_xfip_diff",
    #bullpen stats
    "home_pen_season_era",
    "home_pen_roll14_era",
    "home_pen_season_fip",
    "home_pen_roll14_fip",
    "home_pen_roll14_minus_season_fip",
    "away_pen_season_era",
    "away_pen_roll14_era",
    "away_pen_season_fip",
    "away_pen_roll14_fip",
    "away_pen_roll14_minus_season_fip",	
    # kalshi
    "kalshi_home_implied",
    "kalshi_away_implied",	
    "edge_vs_model"
]
    

data = [c for c in _datapoints if c in view.columns]
ids = [c for c in view.columns if c not in _datapoints]

organized_view = view[data + ids]
organized_view


,game_date,home_team_name,away_team_name,home_wrc_plus,away_wrc_plus,offense_diff,home_offense_platoon,away_offense_platoon,offense_platoon_diff,home_probable_pitcher,home_sp_throws,home_sp_kbb,home_sp_kbb_roll14,home_sp_xfip,home_sp_xfip_roll14,away_probable_pitcher,away_sp_throws,away_sp_kbb,away_sp_kbb_roll14,away_sp_xfip,sp_xfip_diff,home_pen_season_era,home_pen_roll14_era,home_pen_season_fip,home_pen_roll14_fip,home_pen_roll14_minus_season_fip,away_pen_season_era,away_pen_roll14_era,away_pen_season_fip,away_pen_roll14_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,game_pk,detailed_state,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,sp_kbb_diff,away_sp_xfip_roll14,stats_season
637,2026-04-15,Athletics,Texas Rangers,92.871749,97.722193,-4.850445,97.100094,104.536950,-7.436857,J.T. Ginn,R,11.904762,6.250000,3.645455,6.700000,Kumar Rocker,R,11.363636,8.695652,5.000000,-1.354545,4.534351,5.333333,4.588550,4.803704,0.215154,2.006757,2.482759,3.809459,4.513793,0.704334,0.535,0.465,NaN,825021,Scheduled,133,140,NaN,NaN,669372,677958,NaN,True,2026,2026,0.541126,2.500000,2026
632,2026-04-15,Atlanta Braves,Miami Marlins,111.417566,104.284559,7.133007,111.131899,110.991581,0.140318,Bryce Elder,R,16.176471,29.166667,2.873585,1.242857,Chris Paddack,R,13.235294,0.000000,5.077273,-2.203688,0.801980,0.760563,2.238614,2.128169,-0.110445,3.532710,5.571429,3.156075,4.433333,1.277259,0.605,0.395,NaN,824936,Pre-Game,144,146,0.0,0.0,693821,663978,NaN,True,2026,2026,2.941176,3.957143,2026
624,2026-04-15,Baltimore Orioles,Arizona Diamondbacks,103.713919,98.007513,5.706405,118.623019,92.609916,26.013103,Kyle Bradish,R,12.500000,16.666667,3.150847,2.211111,Eduardo Rodriguez,L,5.208333,7.692308,4.186957,-1.036109,3.731707,2.142857,3.563415,3.861905,0.298490,5.360294,4.500000,4.335294,4.100000,-0.235294,0.005,0.995,NaN,824855,Final,110,109,5.0,8.0,680694,593958,0.0,True,2026,2026,7.291667,2.671429,2026
633,2026-04-15,Chicago White Sox,Tampa Bay Rays,83.741500,103.143278,-19.401778,79.700655,103.414406,-23.713751,Sean Burke,R,19.354839,19.047619,1.900000,2.100000,Jesse Scholtens,R,17.647059,17.647059,2.028571,-0.128571,6.590551,3.648649,5.320472,4.600000,-0.720472,6.982759,6.652174,5.841379,5.839130,-0.002249,0.495,0.505,NaN,824615,Pre-Game,145,139,0.0,0.0,680732,669947,NaN,True,2026,2026,1.707780,2.028571,2026
629,2026-04-15,Cincinnati Reds,San Francisco Giants,89.019925,91.445147,-2.425222,83.910196,85.594013,-1.683817,Rhett Lowder,R,7.246377,4.081633,3.651020,3.100000,Tyler Mahle,R,12.500000,8.510638,4.259091,-0.608071,2.842105,2.858824,4.007895,4.052941,0.045046,4.395349,6.287671,4.448837,4.661644,0.212807,0.495,0.505,NaN,824532,Pre-Game,113,137,0.0,0.0,695076,641816,NaN,True,2026,2026,-5.253623,5.350000,2026
627,2026-04-15,Detroit Tigers,Kansas City Royals,98.292834,90.589186,7.703647,99.625819,93.872778,5.753040,Jack Flaherty,R,4.545455,11.111111,5.028571,5.065517,Seth Lugo,R,11.594203,23.809524,2.533962,2.494609,3.701613,2.131579,4.503226,4.007895,-0.495331,5.192308,6.576923,5.292308,5.869231,0.576923,0.545,0.455,NaN,824291,Pre-Game,116,118,0.0,0.0,656427,607625,NaN,True,2026,2026,-7.048748,1.500000,2026
635,2026-04-15,Houston Astros,Colorado Rockies,114.270769,99.148795,15.121974,119.070091,97.521048,21.549043,Spencer Arrighetti,R,7.051282,NaN,5.505660,NaN,Jose Quintana,L,-10.526316,NaN,4.946154,0.559507,7.006329,8.628866,6.517722,6.965979,0.448258,4.180645,3.306122,4.570968,4.263265,-0.307702,0.625,0.375,NaN,824209,Pre-Game,117,115,0.0,0.0,681293,500779,NaN,True,2025,2026,17.577598,NaN,2026
638,2026-04-15,Los Angeles Dodgers,New York Mets,119.549194,89.162585,30.386609,115.762395,86.014967,29.747428,Shohei Ohtani,R,8.510638,4.166667,3.016667,2.933333,Clay Holmes,R,5.555556,7.692308,3.822222,-0.805556,2.675676,4.218750,3.289189,4.131250,0.842061,1.711268,1.317073,3.247887,2.8

## Pitch / offense mismatches ("favorites")

Base criteria from `MismatchBreakdown.ipynb` (SP K-BB edge, SP xFIP edge, offense edge), but this notebook now supports **N-of-3 matching**:

- Example default: **at least 2 of 3** core criteria for home-favored or away-favored.
- Optional extras can still be layered on:
  - **Platoon:** `offense_platoon_diff` agrees with favored side.
  - **Bullpen:** favored side has lower `*_pen_roll14_fip`.

The output also includes a simple **`mismatch_score`** so you can rank candidates while we refine a fuller scoring model.

In [13]:
# --- Core thresholds from MismatchBreakdown.ipynb ---
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
MIN_CORE_MATCHES = 2  # out of 3 core checks (2 = "most", 3 = strict old behavior)

# --- Optional: platoon + bullpen roll14 (lower FIP = better pen) ---
USE_PLATOON_EXTRA = True
PLATOON_MIN = 0.0  # require |offense_platoon_diff| > this when extras on
USE_PEN_EXTRA = True
PEN_ROLL_MIN_GAP = 0.0  # favored pen FIP must be lower by this much vs opponent pen

s = sub.copy()
req = {"sp_kbb_diff", "sp_xfip_diff", "offense_diff"}
missing = req - set(s.columns)
if missing:
    raise ValueError(f"Missing columns for mismatch filter: {missing}")

# Core directional checks for each side
home_core_kbb = s["sp_kbb_diff"] > SP_KBB_ABS
home_core_xfip = s["sp_xfip_diff"] < -SP_XFIP_ABS
home_core_off = s["offense_diff"] > OFFENSE_ABS
home_core_n = home_core_kbb.astype(int) + home_core_xfip.astype(int) + home_core_off.astype(int)

away_core_kbb = s["sp_kbb_diff"] < -SP_KBB_ABS
away_core_xfip = s["sp_xfip_diff"] > SP_XFIP_ABS
away_core_off = s["offense_diff"] < -OFFENSE_ABS
away_core_n = away_core_kbb.astype(int) + away_core_xfip.astype(int) + away_core_off.astype(int)

home_base = home_core_n >= MIN_CORE_MATCHES
away_base = away_core_n >= MIN_CORE_MATCHES

home_ext = home_base.copy()
away_ext = away_base.copy()

if USE_PLATOON_EXTRA and "offense_platoon_diff" in s.columns:
    po = s["offense_platoon_diff"].fillna(0)
    home_ext = home_ext & (po > PLATOON_MIN)
    away_ext = away_ext & (po < -PLATOON_MIN)

if USE_PEN_EXTRA and {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(s.columns):
    hr = s["home_pen_roll14_fip"]
    ar = s["away_pen_roll14_fip"]
    g = PEN_ROLL_MIN_GAP
    # Home favored: home pen better (lower FIP) than away
    pen_ok_home = (hr + g < ar) | hr.isna() | ar.isna()
    # Away favored: away pen better than home
    pen_ok_away = (ar + g < hr) | hr.isna() | ar.isna()
    home_ext = home_ext & pen_ok_home
    away_ext = away_ext & pen_ok_away

# Keep ALL rows so you can inspect near-misses, and add match flags.
# (Old strict filtering is intentionally disabled.)
favorites = s.copy().sort_values(["game_date", "home_team_name"])

favorites["home_core_matches"] = home_core_n.astype(int)
favorites["away_core_matches"] = away_core_n.astype(int)
favorites["home_base_match"] = home_base
favorites["away_base_match"] = away_base
favorites["home_with_extras"] = home_ext
favorites["away_with_extras"] = away_ext

# Preferred side for scoring / display = stronger core count (tie -> absolute edge blend).
_tie_home = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + (-favorites["sp_xfip_diff"]).abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
_tie_away = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
prefer_home = (favorites["home_core_matches"] > favorites["away_core_matches"]) | (
    (favorites["home_core_matches"] == favorites["away_core_matches"]) & (_tie_home >= _tie_away)
)

favorites["_mismatch_side"] = prefer_home.map({True: "home_favored", False: "away_favored"})
favorites["core_matches"] = favorites[["home_core_matches", "away_core_matches"]].max(axis=1)
favorites["favored_team"] = favorites.apply(
    lambda r: r["home_team_name"] if r["_mismatch_side"] == "home_favored" else r["away_team_name"],
    axis=1,
)

# Basic mismatch score (higher = stronger favorite setup)
favorites["mismatch_score"] = (
    (favorites["sp_kbb_diff"].abs() / SP_KBB_ABS)
    + (favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS)
    + (favorites["offense_diff"].abs() / OFFENSE_ABS)
)
if "offense_platoon_diff" in favorites.columns:
    favorites["mismatch_score"] += 0.35 * (favorites["offense_platoon_diff"].abs() / 10.0)
if {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(favorites.columns):
    pen_gap = (favorites["away_pen_roll14_fip"] - favorites["home_pen_roll14_fip"]).where(
        favorites["_mismatch_side"] == "home_favored",
        favorites["home_pen_roll14_fip"] - favorites["away_pen_roll14_fip"],
    )
    favorites["mismatch_score"] += 0.35 * (pen_gap.fillna(0) / 0.75)

favorites = favorites.sort_values(["mismatch_score", "core_matches", "game_date"], ascending=[False, False, True])

_prio = [
    "game_date",
    "detailed_state",
    "favored_team",
    "_mismatch_side",
    "core_matches",
    "mismatch_score",
    "home_team_name",
    "away_team_name",
    "sp_kbb_diff",
    "sp_xfip_diff",
    "offense_diff",
    "offense_platoon_diff",
    "home_offense_platoon",
    "away_offense_platoon",
    "home_pen_roll14_fip",
    "away_pen_roll14_fip",
    "home_pen_season_fip",
    "away_pen_season_fip",
    "home_probable_pitcher",
    "away_probable_pitcher",
]
_prio = [c for c in _prio if c in favorites.columns]
_rest = [c for c in favorites.columns if c not in _prio]
favorites_view = favorites[_prio + _rest]

print(
    f"Core filter (>= {MIN_CORE_MATCHES}/3): home={int(home_base.sum())}, away={int(away_base.sum())} | "
    f"Rows shown: {len(favorites)} (all today/tomorrow rows, sorted by mismatch_score)"
)

# do this if you need to drop rows, for example, rookie pitcher or pitchers not declared
# favorites_view = favorites_view.drop(index=[635, 628])
favorites_view

Core filter (>= 2/3): home=5, away=4 | Rows shown: 15 (all today/tomorrow rows, sorted by mismatch_score)


,game_date,detailed_state,favored_team,_mismatch_side,core_matches,mismatch_score,home_team_name,away_team_name,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_offense_platoon,away_offense_platoon,home_pen_roll14_fip,away_pen_roll14_fip,home_pen_season_fip,away_pen_season_fip,home_probable_pitcher,away_probable_pitcher,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras
628,2026-04-15,Pre-Game,Pittsburgh Pirates,home_favored,2,14.898646,Pittsburgh Pirates,Washington Nationals,18.055556,-2.759740,-5.421085,-18.774432,108.044902,126.819333,3.018919,7.650562,4.258621,6.531655,Mason Montgomery,Jake Irvin,823399,134,120,0.0,0.0,682254,663623,L,R,104.712540,110.133625,30.555556,12.500000,2.554545,5.314286,NaN,True,2026,2026,33.333333,9.090909,5.671429,2.600000,3.910345,6.215827,4.013514,7.887640,-1.239702,1.118907,0.615,0.385,NaN,2026,2,0,True,False,False,False
634,2026-04-15,Pre-Game,Toronto Blue Jays,away_favored,2,13.310959,Milwaukee Brewers,Toronto Blue Jays,-21.451566,2.635135,3.994484,12.628625,112.675398,100.046773,3.963014,3.858621,3.270213,3.671429,Chad Patrick,Dylan Cease,823803,158,141,0.0,0.0,694477,656302,R,R,104.997860,101.003376,3.921569,25.373134,4.235135,1.600000,NaN,True,2026,2026,0.000000,15.555556,3.700000,2.671429,2.106383,4.959184,2.958904,4.344828,0.692801,0.187192,0.485,0.515,NaN,2026,0,2,False,True,False,False
631,2026-04-15,Pre-Game,Los Angeles Angels,away_favored,2,11.518533,New York Yankees,Los Angeles Angels,-6.652047,4.710000,-5.278425,8.699719,107.202993,98.503274,2.013043,4.051220,2.000000,3.803448,Luis Gil,Jack Kochanowicz,823561,147,108,0.0,0.0,661563,686799,R,R,98.007513,103.285938,-5.263158,1.388889,8.350000,3.640000,NaN,True,2026,2026,NaN,21.739130,NaN,1.688235,2.700000,3.351724,4.304348,3.951220,0.013043,0.247771,0.645,0.355,NaN,2026,0,2,False,True,False,False
635,2026-04-15,Pre-Game,Houston Astros,home_favored,2,7.983360,Houston Astros,Colorado Rockies,17.577598,0.559507,15.121974,21.549043,119.070091,97.521048,6.965979,4.263265,6.517722,4.570968,Spencer Arrighetti,Jose Quintana,824209,117,115,0.0,0.0,681293,500779,R,L,114.270769,99.148795,7.051282,-10.526316,5.505660,4.946154,NaN,True,2025,2026,NaN,NaN,NaN,NaN,7.006329,4.180645,8.628866,3.306122,0.448258,-0.307702,0.625,0.375,NaN,2026,2,1,True,False,False,False
626,2026-04-15,Final,Boston Red Sox,away_favored,2,7.886064,Minnesota Twins,Boston Red Sox,-7.556936,2.010115,7.846307,9.917498,104.912821,94.995323,3.931325,3.470370,3.824138,4.186614,Simeon Woods Richardson,Connelly Early,823722,142,111,5.0,9.0,680573,813349,R,L,104.427220,96.580912,4.347826,11.904762,5.567742,3.557627,0.0,True,2026,2026,7.692308,0.000000,4.750000,4.100000,3.956897,2.976378,4.554217,2.666667,0.107187,-0.716244,0.005,0.995,NaN,2026,0,2,False,True,False,False
627,2026-04-15,Pre-Game,Kansas City Royals,away_favored,2,7.441899,Detroit Tigers,Kansas City Royals,-7.048748,2.494609,7.703647,5.753040,99.625819,93.872778,4.007895,5.869231,4.503226,5.292308,Jack Flaherty,Seth Lugo,824291,116,118,0.0,0.0,656427,607625,R,R,98.292834,90.589186,4.545455,11.594203,5.028571,2.533962,NaN,True,2026,2026,11.111111,23.809524,5.065517,1.500000,3.701613,5.192308,2.131579,6.576923,-0.495331,0.576923,0.545,0.455,NaN,2026,0,2,False,True,False,False
632,2026-04-15,Pre-Game,Atlanta Braves,home_favored,1,7.181723,Atlanta Braves,Miami Marlins,2.941176,-2.203688,

In [14]:
import numpy as np
import pandas as pd

s = favorites_view.copy()

# =========================================================
# V6 MODEL — SLATE-NORMALIZED EDGE SYSTEM
# =========================================================

# =========================================================
# 1. NORMALIZED SIGNALS (STABLE SCALING)
# =========================================================

SP_KBB_ABS = 2.5
SP_XFIP_ABS = 0.75
OFFENSE_ABS = 5.0
PLATOON_ABS = 12.0
PEN_ABS = 1.5

kbb_norm = s["sp_kbb_diff"] / SP_KBB_ABS
xfip_norm = -s["sp_xfip_diff"] / SP_XFIP_ABS
off_norm = s["offense_diff"] / OFFENSE_ABS
platoon_norm = s["offense_platoon_diff"].fillna(0) / PLATOON_ABS

pen_gap = s["away_pen_roll14_fip"] - s["home_pen_roll14_fip"]
pen_norm = pen_gap / PEN_ABS

# =========================================================
# 2. STRUCTURED SIGNAL COMPONENTS
# =========================================================

sp_vector = (0.80 * kbb_norm) + (0.20 * xfip_norm)
off_vector = off_norm + (0.15 * platoon_norm)
pen_vector = 0.5 * pen_norm

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])

sp_vector = np.clip(sp_vector, -2, 2)
off_vector = np.clip(off_vector, -2, 2)
pen_vector = np.clip(pen_vector, -2, 2)

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])

# =========================================================
# 3. EDGE CONSTRUCTION
# =========================================================

sign_matrix = np.sign(signal_matrix)
mag_matrix = np.abs(signal_matrix)

mean_sign = np.mean(sign_matrix, axis=0)
mean_mag = np.mean(mag_matrix, axis=0)

raw_edge = mean_sign * mean_mag

# =========================================================
# 4. CONFIDENCE STRUCTURE
# =========================================================

agreement = 1 - np.mean(np.var(sign_matrix, axis=0))
direction_purity = np.abs(mean_sign)
mag_consistency = 1 / (1 + np.std(signal_matrix, axis=0))

coherence_score = (
    0.45 * agreement +
    0.30 * direction_purity +
    0.25 * mag_consistency
)

instability = np.std(signal_matrix, axis=0)
instability_penalty = 1 / (1 + instability)

risk_penalty = np.tanh(
    0.35 * (np.sign(sp_vector) != np.sign(off_vector)).astype(int) +
    0.45 * instability +
    0.20 * np.abs(sp_vector - off_vector)
)

trap_score = np.abs(raw_edge) * (1 - coherence_score)

quality_score = (
    raw_edge *
    coherence_score *
    instability_penalty *
    (1 / (1 + risk_penalty))
)

risk_adjusted_edge = quality_score * np.exp(-trap_score)

# =========================================================
# 5. SLATE NORMALIZATION
# =========================================================

s["edge_rank_pct"] = risk_adjusted_edge.rank(pct=True)
s["edge_z"] = (risk_adjusted_edge - risk_adjusted_edge.mean()) / (risk_adjusted_edge.std() + 1e-9)

# =========================================================
# 6. DECISION LAYER (SLATE RELATIVE)
# =========================================================

s["decision"] = np.select(
    [
        s["edge_rank_pct"] >= 0.90,
        s["edge_rank_pct"] >= 0.75
    ],
    ["BET", "LEAN"],
    default="NO BET"
)

# =========================================================
# 7. TIERS
# =========================================================

s["tier"] = np.select(
    [
        (s["decision"] == "BET") & (s["edge_rank_pct"] >= 0.97),
        (s["decision"] == "BET")
    ],
    ["A (Strong Bet)", "B (Playable Bet)"],
    default=np.where(
        s["decision"] == "LEAN",
        "C (Lean)",
        "D (Pass)"
    )
)

# =========================================================
# 8. TEAM LOGIC
# =========================================================

prefer_home = sp_vector >= 0

s["favored_team"] = np.where(
    prefer_home,
    s["home_team_name"],
    s["away_team_name"]
)

s["_mismatch_side"] = np.where(
    prefer_home,
    "home_favored",
    "away_favored"
)

# =========================================================
# 9. INTERPRETABILITY
# =========================================================

s["signal_agreement"] = np.sum(sign_matrix > 0, axis=0)
s["signal_conflict"] = np.sum(sign_matrix < 0, axis=0)
s["direction_conflict"] = (np.sum(sign_matrix < 0, axis=0) > 0).astype(int)
s["instability"] = instability

s["coherence_score"] = coherence_score
s["quality_score"] = quality_score
s["risk_adjusted_edge"] = risk_adjusted_edge
s["trap_score"] = trap_score

# =========================================================
# 10. FEATURES
# =========================================================

s["core_matches"] = (
    (np.abs(kbb_norm) > 1).astype(int) +
    (np.abs(xfip_norm) > 1).astype(int) +
    (np.abs(off_norm) > 1).astype(int)
)

# =========================================================
# 11. MARKET
# =========================================================

if "kalshi_home_implied" in s.columns:
    s["implied_prob"] = np.where(
        prefer_home,
        s["kalshi_home_implied"],
        s["kalshi_away_implied"]
    )
    s["market_edge"] = risk_adjusted_edge - s["implied_prob"]

# =========================================================
# 12. SORT
# =========================================================

s = s.sort_values(["risk_adjusted_edge", "coherence_score"], ascending=[False, False])

# =========================================================
# 13. OUTPUT
# =========================================================

priority_cols = [
    "game_date",
    "home_team_name",
    "away_team_name",
    "favored_team",
    "_mismatch_side",
    "risk_adjusted_edge",
    "edge_rank_pct",
    "edge_z",
    "decision",
    "tier",
    "coherence_score",
    "trap_score",
    "instability",
    "signal_agreement",
    "signal_conflict",
    "direction_conflict",
    "core_matches",
    "home_probable_pitcher",
    "away_probable_pitcher",
    "sp_kbb_diff",
    "sp_xfip_diff",
    "offense_diff",
    "offense_platoon_diff",
    "home_pen_roll14_fip",
    "away_pen_roll14_fip"
]

priority_cols = [c for c in priority_cols if c in s.columns]
rest_cols = [c for c in s.columns if c not in priority_cols]

final_view = s[priority_cols + rest_cols]

print(f"V6 complete — games evaluated: {len(final_view)}")
final_view


V6 complete — games evaluated: 15


,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,risk_adjusted_edge,edge_rank_pct,edge_z,decision,tier,coherence_score,trap_score,instability,signal_agreement,signal_conflict,direction_conflict,core_matches,home_probable_pitcher,away_probable_pitcher,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_pen_roll14_fip,away_pen_roll14_fip,detailed_state,mismatch_score,home_offense_platoon,away_offense_platoon,home_pen_season_fip,away_pen_season_fip,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,quality_score,implied_prob,market_edge
632,2026-04-15,Atlanta Braves,Miami Marlins,Atlanta Braves,home_favored,0.281619,1.000000,3.064989,BET,A (Strong Bet),0.590278,0.508816,0.337296,3,0,0,3,Bryce Elder,Chris Paddack,2.941176,-2.203688,7.133007,0.140318,2.128169,4.433333,Pre-Game,7.181723,111.131899,110.991581,2.238614,3.156075,824936,144,146,0.0,0.0,693821,663978,R,R,111.417566,104.284559,16.176471,13.235294,2.873585,5.077273,NaN,True,2026,2026,29.166667,0.000000,1.242857,3.957143,0.801980,3.532710,0.760563,5.571429,-0.110445,1.277259,0.605,0.395,NaN,2026,1,0,False,False,False,False,0.468423,0.605,-0.323381
624,2026-04-15,Baltimore Orioles,Arizona Diamondbacks,Baltimore Orioles,home_favored,0.142961,0.933333,1.388366,BET,B (Playable Bet),0.541494,0.541926,0.809493,3,0,0,3,Kyle Bradish,Eduardo Rodriguez,7.291667,-1.036109,5.706405,26.013103,3.861905,4.100000,Final,6.094984,118.623019,92.609916,3.563415,4.335294,824855,110,109,5.0,8.0,680694,593958,R,L,103.713919,98.007513,12.500000,5.208333,3.150847,4.186957,0.0,True,2026,2026,16.666667,7.692308,2.211111,2.671429,3.731707,5.360294,2.142857,4.500000,0.298490,-0.235294,0.005,0.995,NaN,2026,2,0,True,False,True,False,0.245794,0.005,0.137961
638,2026-04-15,Los Angeles Dodgers,New York Mets,Los Angeles Dodgers,home_favored,0.032028,0.866667,0.046994,LEAN,C (Lean),0.327543,0.269114,1.012723,2,1,1,3,Shohei Ohtani,Clay Holmes,2.955083,-0.805556,30.386609,29.747428,4.131250,2.807317,Pre-Game,6.058124,115.762395,86.014967,3.289189,3.247887,823963,119,121,NaN,NaN,660271,605280,R,R,119.549194,89.162585,8.510638,5.555556,3.016667,3.822222,NaN,True,2026,2026,4.166667,7.692308,2.933333,2.814286,2.675676,1.711268,4.218750,1.317073,0.842061,-0.440570,0.645,0.355,NaN,2026,2,0,True,False,False,False,0.041918,0.645,-0.612972
635,2026-04-15,Houston Astros,Colorado Rockies,Houston Astros,home_favored,0.031509,0.800000,0.040715,LEAN,C (Lean),0.308930,0.376319,1.367500,2,1,1,2,Spencer Arrighetti,Jose Quintana,17.577598,0.559507,15.121974,21.549043,6.965979,4.263265,Pre-Game,7.983360,119.070091,97.521048,6.517722,4.570968,824209,117,115,0.0,0.0,681293,500779,R,L,114.270769,99.148795,7.051282,-10.526316,5.505660,4.946154,NaN,True,2025,2026,NaN,NaN,NaN,NaN,7.006329,4.180645,8.628866,3.306122,0.448258,-0.307702,0.625,0.375,NaN,2026,2,1,True,False,False,False,0.045905,0.625,-0.593491
630,2026-04-15,Philadelphia Phillies,Chicago Cubs,Philadelphia Phillies,home_favored,0.027231,0.733333,-0.011010,NO BET,D (Pass),0.336110,0.231452,0.882860,2,1,1,1,Jesús Luzardo,Shota Imanaga,4.319249,0.129808,-1.426601,-25.036014,1.721622,5.297183,Pre-Game,4.386881,80.025831,105.061845,2.332558,5.375000,823477,143,112,0.0,0.0,666200,684007,L,L,99.434115,100.860716,30.985915,26.666667,2.292308,2.162500,NaN,True,2026,2026,45.833333,15.789474,-0.200000,2.100000,4.186047,4.275000,2.5540

In [15]:
def decision_layer(df):

    conditions = [
        # BET (strong + clean)
        (
            (df["risk_adjusted_edge"] > 1.0) &
            (df["coherence_score"] > 0.60) &
            (df["instability"] < 1.2)
        ),

        # LEAN (some edge but imperfect structure)
        (
            (df["risk_adjusted_edge"] > 0.5) &
            (df["coherence_score"] > 0.45)
        ),

        # NO BET (everything else)
    ]

    choices = ["BET", "LEAN"]

    df["decision"] = np.select(conditions, choices, default="NO BET")

    return df

decision_layer(final_view)

,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,risk_adjusted_edge,edge_rank_pct,edge_z,decision,tier,coherence_score,trap_score,instability,signal_agreement,signal_conflict,direction_conflict,core_matches,home_probable_pitcher,away_probable_pitcher,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_pen_roll14_fip,away_pen_roll14_fip,detailed_state,mismatch_score,home_offense_platoon,away_offense_platoon,home_pen_season_fip,away_pen_season_fip,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,quality_score,implied_prob,market_edge
632,2026-04-15,Atlanta Braves,Miami Marlins,Atlanta Braves,home_favored,0.281619,1.000000,3.064989,NO BET,A (Strong Bet),0.590278,0.508816,0.337296,3,0,0,3,Bryce Elder,Chris Paddack,2.941176,-2.203688,7.133007,0.140318,2.128169,4.433333,Pre-Game,7.181723,111.131899,110.991581,2.238614,3.156075,824936,144,146,0.0,0.0,693821,663978,R,R,111.417566,104.284559,16.176471,13.235294,2.873585,5.077273,NaN,True,2026,2026,29.166667,0.000000,1.242857,3.957143,0.801980,3.532710,0.760563,5.571429,-0.110445,1.277259,0.605,0.395,NaN,2026,1,0,False,False,False,False,0.468423,0.605,-0.323381
624,2026-04-15,Baltimore Orioles,Arizona Diamondbacks,Baltimore Orioles,home_favored,0.142961,0.933333,1.388366,NO BET,B (Playable Bet),0.541494,0.541926,0.809493,3,0,0,3,Kyle Bradish,Eduardo Rodriguez,7.291667,-1.036109,5.706405,26.013103,3.861905,4.100000,Final,6.094984,118.623019,92.609916,3.563415,4.335294,824855,110,109,5.0,8.0,680694,593958,R,L,103.713919,98.007513,12.500000,5.208333,3.150847,4.186957,0.0,True,2026,2026,16.666667,7.692308,2.211111,2.671429,3.731707,5.360294,2.142857,4.500000,0.298490,-0.235294,0.005,0.995,NaN,2026,2,0,True,False,True,False,0.245794,0.005,0.137961
638,2026-04-15,Los Angeles Dodgers,New York Mets,Los Angeles Dodgers,home_favored,0.032028,0.866667,0.046994,NO BET,C (Lean),0.327543,0.269114,1.012723,2,1,1,3,Shohei Ohtani,Clay Holmes,2.955083,-0.805556,30.386609,29.747428,4.131250,2.807317,Pre-Game,6.058124,115.762395,86.014967,3.289189,3.247887,823963,119,121,NaN,NaN,660271,605280,R,R,119.549194,89.162585,8.510638,5.555556,3.016667,3.822222,NaN,True,2026,2026,4.166667,7.692308,2.933333,2.814286,2.675676,1.711268,4.218750,1.317073,0.842061,-0.440570,0.645,0.355,NaN,2026,2,0,True,False,False,False,0.041918,0.645,-0.612972
635,2026-04-15,Houston Astros,Colorado Rockies,Houston Astros,home_favored,0.031509,0.800000,0.040715,NO BET,C (Lean),0.308930,0.376319,1.367500,2,1,1,2,Spencer Arrighetti,Jose Quintana,17.577598,0.559507,15.121974,21.549043,6.965979,4.263265,Pre-Game,7.983360,119.070091,97.521048,6.517722,4.570968,824209,117,115,0.0,0.0,681293,500779,R,L,114.270769,99.148795,7.051282,-10.526316,5.505660,4.946154,NaN,True,2025,2026,NaN,NaN,NaN,NaN,7.006329,4.180645,8.628866,3.306122,0.448258,-0.307702,0.625,0.375,NaN,2026,2,1,True,False,False,False,0.045905,0.625,-0.593491
630,2026-04-15,Philadelphia Phillies,Chicago Cubs,Philadelphia Phillies,home_favored,0.027231,0.733333,-0.011010,NO BET,D (Pass),0.336110,0.231452,0.882860,2,1,1,1,Jesús Luzardo,Shota Imanaga,4.319249,0.129808,-1.426601,-25.036014,1.721622,5.297183,Pre-Game,4.386881,80.025831,105.061845,2.332558,5.375000,823477,143,112,0.0,0.0,666200,684007,L,L,99.434115,100.860716,30.985915,26.666667,2.292308,2.162500,NaN,True,2026,2026,45.833333,15.789474,-0.200000,2.100000,4.186047,4.275

In [16]:
def confidence_tier(df):

    df["tier"] = np.where(
        (df["decision"] == "BET") & (df["coherence_score"] > 0.75),
        "A (Strong Bet)",
        np.where(
            (df["decision"] == "BET"),
            "B (Playable Bet)",
            np.where(
                (df["decision"] == "LEAN"),
                "C (Lean Only)",
                "D (Pass)"
            )
        )
    )

    return df

confidence_tier(final_view)

,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,risk_adjusted_edge,edge_rank_pct,edge_z,decision,tier,coherence_score,trap_score,instability,signal_agreement,signal_conflict,direction_conflict,core_matches,home_probable_pitcher,away_probable_pitcher,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,home_pen_roll14_fip,away_pen_roll14_fip,detailed_state,mismatch_score,home_offense_platoon,away_offense_platoon,home_pen_season_fip,away_pen_season_fip,game_pk,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_season_stats_complete,home_sp_stats_season,away_sp_stats_season,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_era,away_pen_season_era,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,quality_score,implied_prob,market_edge
632,2026-04-15,Atlanta Braves,Miami Marlins,Atlanta Braves,home_favored,0.281619,1.000000,3.064989,NO BET,D (Pass),0.590278,0.508816,0.337296,3,0,0,3,Bryce Elder,Chris Paddack,2.941176,-2.203688,7.133007,0.140318,2.128169,4.433333,Pre-Game,7.181723,111.131899,110.991581,2.238614,3.156075,824936,144,146,0.0,0.0,693821,663978,R,R,111.417566,104.284559,16.176471,13.235294,2.873585,5.077273,NaN,True,2026,2026,29.166667,0.000000,1.242857,3.957143,0.801980,3.532710,0.760563,5.571429,-0.110445,1.277259,0.605,0.395,NaN,2026,1,0,False,False,False,False,0.468423,0.605,-0.323381
624,2026-04-15,Baltimore Orioles,Arizona Diamondbacks,Baltimore Orioles,home_favored,0.142961,0.933333,1.388366,NO BET,D (Pass),0.541494,0.541926,0.809493,3,0,0,3,Kyle Bradish,Eduardo Rodriguez,7.291667,-1.036109,5.706405,26.013103,3.861905,4.100000,Final,6.094984,118.623019,92.609916,3.563415,4.335294,824855,110,109,5.0,8.0,680694,593958,R,L,103.713919,98.007513,12.500000,5.208333,3.150847,4.186957,0.0,True,2026,2026,16.666667,7.692308,2.211111,2.671429,3.731707,5.360294,2.142857,4.500000,0.298490,-0.235294,0.005,0.995,NaN,2026,2,0,True,False,True,False,0.245794,0.005,0.137961
638,2026-04-15,Los Angeles Dodgers,New York Mets,Los Angeles Dodgers,home_favored,0.032028,0.866667,0.046994,NO BET,D (Pass),0.327543,0.269114,1.012723,2,1,1,3,Shohei Ohtani,Clay Holmes,2.955083,-0.805556,30.386609,29.747428,4.131250,2.807317,Pre-Game,6.058124,115.762395,86.014967,3.289189,3.247887,823963,119,121,NaN,NaN,660271,605280,R,R,119.549194,89.162585,8.510638,5.555556,3.016667,3.822222,NaN,True,2026,2026,4.166667,7.692308,2.933333,2.814286,2.675676,1.711268,4.218750,1.317073,0.842061,-0.440570,0.645,0.355,NaN,2026,2,0,True,False,False,False,0.041918,0.645,-0.612972
635,2026-04-15,Houston Astros,Colorado Rockies,Houston Astros,home_favored,0.031509,0.800000,0.040715,NO BET,D (Pass),0.308930,0.376319,1.367500,2,1,1,2,Spencer Arrighetti,Jose Quintana,17.577598,0.559507,15.121974,21.549043,6.965979,4.263265,Pre-Game,7.983360,119.070091,97.521048,6.517722,4.570968,824209,117,115,0.0,0.0,681293,500779,R,L,114.270769,99.148795,7.051282,-10.526316,5.505660,4.946154,NaN,True,2025,2026,NaN,NaN,NaN,NaN,7.006329,4.180645,8.628866,3.306122,0.448258,-0.307702,0.625,0.375,NaN,2026,2,1,True,False,False,False,0.045905,0.625,-0.593491
630,2026-04-15,Philadelphia Phillies,Chicago Cubs,Philadelphia Phillies,home_favored,0.027231,0.733333,-0.011010,NO BET,D (Pass),0.336110,0.231452,0.882860,2,1,1,1,Jesús Luzardo,Shota Imanaga,4.319249,0.129808,-1.426601,-25.036014,1.721622,5.297183,Pre-Game,4.386881,80.025831,105.061845,2.332558,5.375000,823477,143,112,0.0,0.0,666200,684007,L,L,99.434115,100.860716,30.985915,26.666667,2.292308,2.162500,NaN,True,2026,2026,45.833333,15.789474,-0.200000,2.100000,4.186047,4.275000,2.554054,4